In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import h5py as h5
from cosmopower import cosmopower_NN
import pickle

In [ ]:
#Here we initialize the network and load the saved model

modes = np.arange(80)
cp_nn = cosmopower_NN(parameters=['Omega_m', 's8'], 
                      modes=modes, 
                      n_hidden = [64, 256, 1024, 1024, 512, 256, 128],
                      verbose=True,
                     )

model_filename = "map3_model_11Apr_001"
cp_nn.restore(model_filename)

In [ ]:
#this function is to rescale the input params (omega_m and s8)

def rescale_params(params, scale):
    
    params_rescaled = np.zeros_like(params)
    params_rescaled[:,0] = (params[:,0]-scale['Omega_m']['small'])/(scale['Omega_m']['large']-scale['Omega_m']['small'])
    params_rescaled[:,1] = (params[:,1]-scale['s8']['small'])/(scale['s8']['large']-scale['s8']['small'])
    
    return(params_rescaled)

In [ ]:
#this function is to rescale the output map3 values

def post_process(array, scale):
    
    out_array = np.zeros_like(array)
    samples = len(array)
    maxv = scale['large']
    minv = scale['small']
    
    for i in range(len(array[0])):
        tmp = array[:,i]*(maxv[i]-minv[i])+minv[i]
        out_array[:,i] = 10**(tmp)
    
    return(out_array)

In [ ]:
#here we load our test parameters, along with their true values

test_params = np.load('test_params_map3.npy')
test_true_vals = np.load('test_true_vals_map3.npy')

In [ ]:
#here we load the rescaling values for the input params and for the output data

rescaling_filename = "rescaling_for_map3_network.pkl"
with open(rescaling_filename, 'rb') as file:
    rescaling_values = pickle.load(file)

rescaling_params = rescaling_values['params']
rescaling_features = rescaling_values['features']

In [ ]:
#here we rescale our test parameters and make the predictions with our saved model

params_for_network = rescale_params(test_params, rescaling_params)
test_params_dict = {'Omega_m': params_for_network[:,0],
          's8': params_for_network[:,1]}
    
predictions = cp_nn.predictions_np(test_params_dict)
predictions_rescaled = post_process(predictions, rescaling_features)

In [ ]:
#here we can make plots and analyze the data

plt.title("map3 emulator performance, element 0 of dv")
plt.scatter(test_true_vals[:,0], predictions_rescaled[:,0], s=2)
plt.xlabel("True")
plt.ylabel("Predicted")
plt.show()

In [ ]:
plt.title("map3 emulator performance, all elements of dv")
plt.scatter(test_true_vals.flatten(), predictions_rescaled.flatten(), s=2)
plt.xlabel("True")
plt.ylabel("Predicted")
plt.show()